# TripoSG - 3D Model Generation for OpenTTS

Generate 3D models from images for use in the OpenTTS tabletop simulator.

**Based on [TripoSG](https://github.com/VAST-AI-Research/TripoSG) by VAST-AI-Research (MIT License)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DutchMaxwell/openTTS/blob/main/tools/3d-generation/triposg_colab.ipynb)

## 1. GPU Check
Verify GPU is available (T4 or better recommended)

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU")

## 2. Install TripoSG
Clone repository and install dependencies (~2-3 min)

In [ ]:
# Clone TripoSG
!git clone https://github.com/VAST-AI-Research/TripoSG.git
%cd TripoSG

# Install dependencies
!pip install -q -r requirements.txt

print("\n✅ TripoSG installed successfully!")

## 3. Mount Google Drive
For persistent storage of input images and output models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create folders in Drive
import os
BASE_DIR = '/content/drive/MyDrive/OpenTTS_3D'
INPUT_DIR = f'{BASE_DIR}/input'
OUTPUT_DIR = f'{BASE_DIR}/output'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n✅ Google Drive mounted!")
print(f"📁 Input folder: {INPUT_DIR}")
print(f"📁 Output folder: {OUTPUT_DIR}")
print(f"\nUpload your images to: Google Drive > MyDrive > OpenTTS_3D > input")

## 4. Single Image Test
Test with one image first

In [ ]:
from pathlib import Path
import time

# Find first image in input folder
input_images = list(Path(INPUT_DIR).glob('*.png')) + list(Path(INPUT_DIR).glob('*.jpg'))

if not input_images:
    print("❌ No images found in input folder!")
    print(f"Please upload images to: {INPUT_DIR}")
else:
    test_image = input_images[0]
    output_path = f"{OUTPUT_DIR}/{test_image.stem}.glb"
    
    print(f"🎨 Processing: {test_image.name}")
    start = time.time()
    
    !python -m scripts.inference_triposg \
        --image-input "{test_image}" \
        --output-path "{output_path}" \
        --faces 10000
    
    elapsed = time.time() - start
    print(f"\n✅ Done in {elapsed:.1f}s")
    print(f"📦 Output: {output_path}")

## 5. Batch Processing
Process all images in the input folder

In [ ]:
from pathlib import Path
import time

# Settings
FACES = 10000  # Polygon count (5000=low, 10000=medium, 20000=high)
SKIP_EXISTING = True  # Skip if output already exists

# Find all images
input_images = list(Path(INPUT_DIR).glob('*.png')) + list(Path(INPUT_DIR).glob('*.jpg'))
print(f"Found {len(input_images)} images to process\n")

results = {'success': 0, 'failed': 0, 'skipped': 0}
total_start = time.time()

for i, img_path in enumerate(input_images, 1):
    output_path = f"{OUTPUT_DIR}/{img_path.stem}.glb"
    
    # Skip existing
    if SKIP_EXISTING and Path(output_path).exists():
        print(f"[{i}/{len(input_images)}] ⏭️ Skipping {img_path.name} (exists)")
        results['skipped'] += 1
        continue
    
    print(f"[{i}/{len(input_images)}] 🎨 Processing {img_path.name}...")
    start = time.time()
    
    try:
        !python -m scripts.inference_triposg \
            --image-input "{img_path}" \
            --output-path "{output_path}" \
            --faces {FACES} \
            2>/dev/null
        
        if Path(output_path).exists():
            elapsed = time.time() - start
            print(f"    ✅ Done in {elapsed:.1f}s")
            results['success'] += 1
        else:
            print(f"    ❌ Failed")
            results['failed'] += 1
    except Exception as e:
        print(f"    ❌ Error: {e}")
        results['failed'] += 1

total_elapsed = time.time() - total_start
print(f"\n{'='*50}")
print(f"✅ Success: {results['success']}")
print(f"❌ Failed: {results['failed']}")
print(f"⏭️ Skipped: {results['skipped']}")
print(f"⏱️ Total time: {total_elapsed:.1f}s")
print(f"\n📁 Output folder: {OUTPUT_DIR}")

## 6. Download Results
Download generated models directly or copy from Drive

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

# List all generated models
output_files = list(Path(OUTPUT_DIR).glob('*.glb'))
print(f"Generated {len(output_files)} models:\n")

for f in output_files:
    size_mb = f.stat().st_size / 1e6
    print(f"  📦 {f.name} ({size_mb:.1f} MB)")

# Option: Create ZIP for download
if output_files:
    zip_path = f"{BASE_DIR}/models.zip"
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', OUTPUT_DIR)
    print(f"\n📦 Created: {zip_path}")
    print(f"\nDownload from Google Drive or run next cell to download directly.")

In [ ]:
# Direct download (uncomment to use)
# from google.colab import files
# files.download(f"{BASE_DIR}/models.zip")

## 7. Alternative: Upload Images Directly
If you don't want to use Google Drive

In [ ]:
from google.colab import files
import shutil

# Upload files
print("Select images to upload...")
uploaded = files.upload()

# Move to input folder
LOCAL_INPUT = '/content/TripoSG/input'
LOCAL_OUTPUT = '/content/TripoSG/output'
os.makedirs(LOCAL_INPUT, exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

for filename in uploaded.keys():
    shutil.move(filename, f"{LOCAL_INPUT}/{filename}")
    print(f"  ✅ {filename}")

print(f"\nUploaded {len(uploaded)} files to {LOCAL_INPUT}")

## 8. Low-Poly Option
For better game performance (5000 faces)

In [ ]:
from pathlib import Path
import time

# Low-poly settings for game use
FACES_LOWPOLY = 5000
OUTPUT_LOWPOLY = f"{OUTPUT_DIR}/lowpoly"
os.makedirs(OUTPUT_LOWPOLY, exist_ok=True)

input_images = list(Path(INPUT_DIR).glob('*.png')) + list(Path(INPUT_DIR).glob('*.jpg'))

for i, img_path in enumerate(input_images, 1):
    output_path = f"{OUTPUT_LOWPOLY}/{img_path.stem}_lowpoly.glb"
    
    if Path(output_path).exists():
        print(f"[{i}] ⏭️ Skipping {img_path.name}")
        continue
    
    print(f"[{i}] 🎨 Processing {img_path.name} (low-poly)...")
    
    !python -m scripts.inference_triposg \
        --image-input "{img_path}" \
        --output-path "{output_path}" \
        --faces {FACES_LOWPOLY} \
        2>/dev/null
    
    print(f"    ✅ Done")

print(f"\n📁 Low-poly output: {OUTPUT_LOWPOLY}")